# Analisis Kesesuaian Star Rating dan Sentimen Ulasan
## Menggunakan Metode Multinomial Naive Bayes (Premium Version)

Proyek ini bertujuan untuk membangun model klasifikasi sentimen otomatis untuk ulasan aplikasi Google Play Store dan membandingkan hasilnya dengan rating bintang yang diberikan pengguna. Versi ini telah ditingkatkan dengan **Lemmatization**, **Hyperparameter Tuning**, dan **Advanced Visualization**.

### 1. Persiapan Lingkungan dan Import Library
Tahap ini mencakup pemuatan pustaka yang diperlukan untuk manipulasi data, pemrosesan teks, dan pembangunan model machine learning.

In [ ]:
import pandas as pd
import numpy as np
import re
import kagglehub
from kagglehub import KaggleDatasetAdapter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from imblearn.over_sampling import RandomOverSampler

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Set style visualisasi
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 2. Pemuatan Dataset dari Kaggle
Menggunakan `kagglehub` untuk menarik data terbaru secara otomatis dari repositori publik Kaggle.

In [ ]:
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "hassaanmustafavi/google-play-app-reviews-dataset",
  "GooglePlay_App_Data.csv",
)

print(f"Total data: {len(df)}")
df.head()

### 3. Pembersihan dan Transformasi Data
Melakukan pembersihan data mentah, menghapus nilai null, dan memetakan kolom ke format yang lebih konsisten.

In [ ]:
df = df.dropna(subset=['review_description'])
df = df.rename(columns={
    "review_description": "review",
    "rating": "star_rating"
})

# Konversi rating ke sentimen
df['sentiment'] = df['star_rating'].apply(lambda x:
    'positive' if x >= 4 else
    'neutral' if x == 3 else
    'negative'
)

print("Distribusi Sentimen:")
print(df['sentiment'].value_counts())

sns.countplot(x='sentiment', data=df, palette='viridis', order=['negative', 'neutral', 'positive'])
plt.title('Distribusi Sentimen Berdasarkan Rating Bintang')
plt.show()

### 4. Text Preprocessing (Lemmatization)
Langkah ini ditingkatkan dengan **Lemmatization** untuk mengembalikan kata ke bentuk dasarnya, sehingga meningkatkan kepadatan fitur.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase dan hapus karakter non-huruf
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenisasi
    words = text.split()
    
    # Stopwords removal dan Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)

print("Proses preprocessing teks sedang berjalan...")
df['clean_review'] = df['review'].apply(clean_text)

print("\nContoh Perubahan Teks:")
df[['review', 'clean_review']].head()

### 5. Ekstraksi Fitur dengan TF-IDF
Mengubah teks ulasan menjadi representasi vektor numerik menggunakan metode *Term Frequency-Inverse Document Frequency*.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X = vectorizer.fit_transform(df['clean_review'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 6. Menangani Ketidakseimbangan Data (Oversampling)
Menggunakan `RandomOverSampler` agar model dapat mempelajari pola dari kelas minoritas secara adil.

In [ ]:
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

plt.subplot(1, 2, 1)
y_train.value_counts().plot(kind='bar', color='skyblue', title='Sebelum Oversampling')
plt.subplot(1, 2, 2)
pd.Series(y_train_res).value_counts().plot(kind='bar', color='salmon', title='Sesudah Oversampling')
plt.tight_layout()
plt.show()

### 7. Optimasi Model dengan GridSearchCV
Mencari parameter `alpha` terbaik untuk mendapatkan performa klasifikasi yang optimal.

In [ ]:
parameters = {'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]}
grid_search = GridSearchCV(MultinomialNB(), parameters, cv=5, scoring='f1_macro', n_jobs=-1)
grid_search.fit(X_train_res, y_train_res)

print(f"Best Hyperparameter: {grid_search.best_params_}")
model = grid_search.best_estimator_

### 8. Evaluasi Kinerja Model
Menganalisis performa model menggunakan metrik standar dan visualisasi Confusion Matrix.

In [ ]:
y_pred = model.predict(X_test)

print("--- Evaluation Report ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score (Macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['negative', 'neutral', 'positive'], 
            yticklabels=['negative', 'neutral', 'positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix Heatmap')
plt.show()

### 9. Visualisasi Kata Kunci (Word Cloud)
Melihat kata-kata yang paling merepresentasikan setiap kategori sentimen.

In [ ]:
def show_wordcloud(sentiment_type, cmap):
    text = " ".join(df[df['sentiment'] == sentiment_type]['clean_review'])
    wc = WordCloud(width=800, height=400, background_color='white', colormap=cmap).generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(f'Kumpulan Kata Sentimen {sentiment_type.upper()}')
    plt.axis('off')
    plt.show()

show_wordcloud('positive', 'Greens')
show_wordcloud('negative', 'Reds')

### 10. Analisis Kesesuaian & Mismatch
Bagian inti skripsi: menganalisis di mana model berbeda pandangan dengan rating bintang pengguna.

In [ ]:
mismatch_df = pd.DataFrame({
    "Review": df.iloc[y_test.index]['review'],
    "Actual_Rating": df.iloc[y_test.index]['star_rating'],
    "True_Sentiment": y_test.values,
    "Predicted_Sentiment": y_pred
})

mismatch_df['is_correct'] = mismatch_df['True_Sentiment'] == mismatch_df['Predicted_Sentiment']

print(f"Tingkat Kesesuaian Model vs Rating: {mismatch_df['is_correct'].mean():.2%}")

print("\n--- Kasus Ekstrim (Rating 5 tapi Prediksi Negatif) ---")
extreme_mismatch = mismatch_df[(mismatch_df['Actual_Rating'] == 5) & (mismatch_df['Predicted_Sentiment'] == 'negative')]
extreme_mismatch.head(5)

### 11. Penyimpanan Model
Menyimpan model dan vectorizer untuk digunakan kembali (misalnya di aplikasi web).

In [ ]:
import pickle

pickle.dump(model, open("model_naive_bayes_v2.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer_v2.pkl", "wb"))
print("Model berhasil disimpan!")